# Lab 4 — Tracing agentic workflows with self-hosted Langfuse

In **Lab 3** we wrote a propose↔critique graph (`idea_maker` / `idea_hater`) and logged every LLM call to a JSON Lines file via a LangChain callback. That works for one notebook, but it doesn't scale:

- One log file per run; nothing aggregates across runs.
- No UI; you grep or `jq`.
- No cost calculation; you'd have to maintain a price table by hand.
- No way to share runs with collaborators.

**Lab 4** fixes all of that by routing the same LangChain callbacks into [Langfuse](https://langfuse.com) — an open-source LLM-observability platform — running locally on your machine. **The graph code doesn't change**; only the callback handler does.

We also stop hand-rolling agents in the notebook. Lab 4 imports them from a real Python package, **`cmbagent_lg`**, that lives in `~/GitHub/cmbagent_lg`. It's the start of a LangGraph re-implementation of [cmbagent](https://github.com/CMBAgents/cmbagent) deep research — for this lab, just the `planner` ↔ `plan_reviewer` propose-critique loop adapted from Lab 3's `idea_maker` / `idea_hater`.

By the end of this lab you'll be able to:

1. Bring up a self-hosted Langfuse instance with one command.
2. Wire a Langfuse callback into a LangGraph run with one line.
3. Read traces, observations, costs, and latencies in both the UI and via the public API.
4. Aggregate cost / latency / model usage per *agent* using LangChain `tags`.

## Prerequisites

This notebook runs against the **`cmbagent_lg`** package and a local Langfuse stack:

1. `cmbagent_lg` cloned at `~/GitHub/cmbagent_lg` and installed:

    ```bash
    python3.12 -m venv ~/pyvenvs/py312-cmbagent-lg
    source ~/pyvenvs/py312-cmbagent-lg/bin/activate
    pip install -e '~/GitHub/cmbagent_lg[dev]'
    python -m ipykernel install --user --name py312-cmbagent-lg --display-name 'Python 3.12 (cmbagent_lg)'
    ```

2. `langfuse` cloned at `~/GitHub/langfuse`. Start the stack (one-time):

    ```bash
    cd ~/GitHub/langfuse && docker compose up -d
    ```

3. Open `http://localhost:3000`, sign up (any email/password — purely local), create a project, and copy the API keys into `~/GitHub/cmbagent_lg/.env`:

    ```
    GOOGLE_API_KEY=AIza...
    LANGFUSE_HOST=http://localhost:3000
    LANGFUSE_PUBLIC_KEY=pk-lf-...
    LANGFUSE_SECRET_KEY=sk-lf-...
    ```

Pick the **Python 3.12 (cmbagent_lg)** kernel for this notebook.

In [ ]:
from dotenv import load_dotenv

# override=True so .env beats any stale shell exports (LANGFUSE_* in .zshrc, etc.).
load_dotenv('/Users/boris/GitHub/cmbagent_lg/.env', override=True)

import os
for k in ('GOOGLE_API_KEY', 'LANGFUSE_HOST', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY'):
    assert os.environ.get(k), f'missing {k} in .env'
print('env OK')

## 1. Langfuse — the only four concepts you need

| Concept | What it is | Lab 3 equivalent |
| --- | --- | --- |
| **Trace** | One end-to-end run (one `graph.invoke`). Has a UUID, total cost, total latency. | One `*.jsonl` file. |
| **Observation** | A node within a trace. See type table below. | One JSON line. |
| **Tag** | A short string attached to an observation, used for aggregation. We tag every LLM call with the *agent role*: `planner`, `plan_reviewer`, `format_plan`, `format_review`. | A custom field we'd have to invent. |
| **Model** | A name (e.g. `gemini-3.1-flash-lite`) + price entries. Langfuse matches incoming model strings via regex to a price row to compute cost. | We'd have to maintain prices by hand. |

### Observation types

Langfuse's own data model has three primitive types: **`SPAN`** (a unit of timed work), **`GENERATION`** (an LLM call — has `model`, `usage`, `costDetails`), and **`EVENT`** (a zero-duration marker). The LangChain integration emits richer types that Langfuse stores verbatim:

| Type | Meaning | In our setup |
| --- | --- | --- |
| **`GENERATION`** | An LLM call. | Every `_proposer().invoke(...)`, `_critic().invoke(...)`, and formatter call → one GENERATION. With `num_rounds=2` you get **10**: 3 planner + 3 format_plan + 2 plan_reviewer + 2 format_review. |
| **`CHAIN`** | A LangChain `Runnable` execution — a node, a router, an internal `RunnableSequence`, an output parser. | The connective tissue. Names you'll see: `planner`, `format_plan`, `route_after_format_plan`, `RunnableSequence`, `PydanticOutputParser`. |
| `SPAN` / `EVENT` | Langfuse-native primitives for custom instrumentation. | None — we don't emit any. |
| `TOOL` / `RETRIEVER` / `EMBEDDING` / `AGENT` | LangChain-specific types for tool calls, retrieval, embeddings, AgentExecutor runs. | None today; `TOOL` will appear once we add an engineer node that runs code. |

**Practical rule:** `GENERATION` is what you pay for and what `cmbagent-lg-cost` counts. `CHAIN` rows are how LangChain wires the GENERATIONs together — useful for navigating the tree, ignorable for cost. Everything else (SPAN/EVENT/TOOL/...) only appears when something explicitly emits them.

## 2. The wiring — one callback handler, zero graph changes

In Lab 3 we wrote a custom `JsonlLLMLogger(BaseCallbackHandler)`. In Lab 4 we use Langfuse's built-in one:

```python
from langfuse.langchain import CallbackHandler
handler = CallbackHandler()  # reads LANGFUSE_PUBLIC_KEY / SECRET_KEY / HOST from env

graph.invoke({}, context=ctx, config={'callbacks': [handler]})
```

LangChain propagates callbacks automatically into every child run — including the formatter sub-calls inside `with_structured_output`. So one handler captures every LLM call in the trace, with full input, output, latency, tokens, and (if a price is configured) cost.

That's the entire integration. `cmbagent_lg/tracing.py` is a tiny wrapper that fails loudly if keys aren't set:

In [ ]:
from cmbagent_lg.tracing import langfuse_handler

handler = langfuse_handler()
type(handler).__name__, handler.last_trace_id  # no trace yet

## 3. The `cmbagent_lg` package — the graph, lifted out of the notebook

Lab 3 built the propose↔critique graph in-notebook with `idea_maker` / `idea_hater`. For Lab 4 we apply the same pattern to **planning** and ship it as an installable package:

```
cmbagent_lg/
├── context.py     # PlanContext — per-run knobs (task, agents available, etc.)
├── state.py       # PlanState — the mutable bag flowing between nodes
├── schemas.py     # Pydantic Plan + Step + Review (mirror cmbagent's)
├── prompts.py     # YAML loader + schema_field_brief helper
├── nodes.py       # planner / format_plan / plan_reviewer / format_review
├── graph.py       # compiled StateGraph
├── tracing.py     # Langfuse handler factory
├── persistence.py # save_final_plan, save_trace_id, default_work_dir
├── cli.py         # `cmbagent-lg-cost` (per-tag aggregation from langfuse)
└── templates/
    ├── planner.yaml         (vendored from cmbagent)
    └── plan_reviewer.yaml   (vendored from cmbagent)
```

The graph itself is small:

```
START → planner → format_plan → [END if last round else continue]
                                │
                                ▼
                            plan_reviewer → format_review → planner …
```

`num_rounds` counts **review cycles**, so total planner passes = `num_rounds + 1`. The last planner pass has read every prior review (the full transcript is in `state['history']`) and produces the final plan.

In [ ]:
from cmbagent_lg import graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 4. Run it — same `graph.invoke` shape, now traced

Set the per-run context (the LangGraph-native answer to cmbagent's YAML `{improved_main_task}` etc.), attach the handler, invoke. Nothing in the graph or the package changes between runs — context is everything.

In [ ]:
from cmbagent_lg import PlanContext

ctx = PlanContext(
    improved_main_task=(
        'Reconstruct the CMB lensing potential from a single Planck-like temperature '
        'map. We have only the map; no external catalogs.'
    ),
    hardware_constraints='Standard laptop. Single CPU. No GPU. 16 GB RAM.',
    code_execution_timeout=120,
    maximum_number_of_steps_in_plan=5,
    num_rounds=2,  # 2 reviews → 3 planner passes (final plan = v3)
    available_agents=[
        ('engineer',   'Writes and runs Python code: numeric work, data I/O, computations, plotting.'),
        ('researcher', 'Reads, reasons, writes prose; no code execution.'),
    ],
)

result = graph.invoke({}, context=ctx, config={'callbacks': [handler]})

print(f'reviewed rounds:    {len(result["history"])}')
print(f'final plan steps:   {len(result["current_plan"].sub_tasks)}')
print(f'langfuse trace id:  {handler.last_trace_id}')
print(f'open in UI:         http://localhost:3000/trace/{handler.last_trace_id}')

Open the trace URL printed above. You'll see a tree:

- Top-level `LangGraph` span — the full invocation.
- Inside: one **`SPAN`** per node (`planner`, `format_plan`, `plan_reviewer`, `format_review`), in execution order, each with the *full state snapshot* it received as input. (`format_review` sees `raw_plan` and `current_plan` even though it only processes `raw_review` — that's not a bug; every LangGraph node sees the entire state. The LLM call inside only uses what it needs.)
- Inside each node: the `gemini-3.1-flash-lite` **`GENERATION`** span with the actual prompt, response, tokens, latency, and cost.

The same data is queryable via the **public REST API** at `/api/public/traces/{id}` and `/api/public/observations?traceId={id}` — that's what powers the next cell.

## 5. Per-agent aggregation — Langfuse instead of `cost_report.json`

cmbagent's `cost_report.json` shows a table of `agent / model / tokens / cost`. Langfuse can produce the same — we just need each LLM call to be tagged with its agent role.

In `cmbagent_lg/nodes.py`, every LLM call carries a tag:

```python
msg = _proposer().invoke(
    [SystemMessage(system), HumanMessage(user)],
    config={'tags': ['planner']},        # ← here
)
```

The tag propagates into the langfuse `Generation`'s `tags` field. The **`cmbagent-lg-cost`** CLI (installed by `pip install -e .`) queries the trace, groups by tag, and prints a cmbagent-style table:

In [ ]:
trace_id = handler.last_trace_id
!cmbagent-lg-cost {trace_id}

Columns to read carefully:

- **calls** — how many times this agent ran. `planner` = 3 (v1, v2, v3); `format_plan` = 3; `plan_reviewer` = 2 and `format_review` = 2 (because `num_rounds=2`).
- **model** — which Gemini variant served this agent's calls. If you override mid-session (e.g. `os.environ['FORMAT_MODEL'] = 'gemini-3-flash-preview'` and restart the kernel), this column splits.
- **avg_s / min_s / max_s** — the LLM's own latency. Useful to spot the slow agent.
- **$cost** — Langfuse computes this from its **Model Definitions** table. If you see `0.000000`, the model string didn't match any price entry. Fix: Settings → Models → New, with name `gemini-3.1-flash-lite`, match pattern `(?i)^(gemini-3.1-flash-lite)$`, input/output prices from [Google's pricing page](https://ai.google.dev/pricing).

## 6. Local persistence — what we *do* save to disk

Langfuse covers telemetry. Two things still go to local disk:

1. **The final plan** as JSON — downstream control-phase code (cmbagent today, the langgraph port later) consumes it.
2. **The langfuse trace ID** as text — to cross-reference a run's plan with its trace later.

```
work_dir/{run_name}/
├── planning/
│   └── final_plan.json   (shape matches cmbagent's save_final_plan)
└── langfuse_trace_id.txt
```

In [ ]:
from cmbagent_lg import save_final_plan, save_trace_id, default_work_dir

work_dir = default_work_dir()  # work_dir/{YYYY-MM-DD_HH-MM-SS}/ — pass any Path to override
plan_path = save_final_plan(result['current_plan'], work_dir)
tid_path  = save_trace_id(handler.last_trace_id, work_dir)
print(plan_path)
print(tid_path)

Now you can run the cost CLI against the workdir directly — it reads the trace id from the text file:

In [ ]:
!cmbagent-lg-cost {work_dir}

This pattern keeps **language-level orchestration** (Python, this notebook, the `cmbagent_lg` package) decoupled from **observability infrastructure** (Langfuse). Swap the handler for one that ships to Datadog, Honeycomb, a WebSocket, etc., without touching the graph or the package API.

## 7. The architectural payoff — same graph, different team

The graph is compiled once. To run on a different problem with a different agent team, you only change the `PlanContext`. That's the LangGraph-native answer to cmbagent's *"modify YAML and reinitialize agents"*.

The big knobs:

- **`improved_main_task`** — what to plan for.
- **`planner_append_instructions`** / **`plan_reviewer_append_instructions`** — per-task hints injected into the system prompts on top of the base YAML.
- **`available_agents`** — the team. The planner is *forbidden* from emitting any agent name not in this list, enforced by a `Literal[*agent_names]` constructed dynamically inside the formatter.

In [ ]:
ctx2 = PlanContext(
    improved_main_task=(
        'Forecast the next-day return distribution of S&P 500 sector ETFs from yesterday\'s '
        'intraday volatility surface.'
    ),
    hardware_constraints='Standard laptop. No GPU.',
    num_rounds=2,
    available_agents=[
        ('engineer',   'Writes and runs Python code: pandas, numpy, scikit-learn.'),
        ('researcher', 'Reads quantitative-finance literature and writes methodology prose.'),
        ('data_scout', 'Locates and downloads financial datasets (no analysis).'),
    ],
    planner_append_instructions=(
        'Use 5 years of historical data. Prefer interpretable models over deep learning.'
    ),
)

result2 = graph.invoke({}, context=ctx2, config={'callbacks': [handler]})

print(f'final plan steps: {len(result2["current_plan"].sub_tasks)}')
print(f'agents used:      {sorted({s.sub_task_agent for s in result2["current_plan"].sub_tasks})}')
print(f'trace:            http://localhost:3000/trace/{handler.last_trace_id}')

Notice the second trace shows the planner assigning work to `data_scout` — an agent that didn't exist in the first run. The graph code didn't change; only the context did.

This is the architectural difference from cmbagent's YAML approach: there, prompt templates are bound to agent definitions; here, prompts read context at runtime, and a single compiled graph generalizes.

## 8. Recap

**Concepts**

- A **trace** is one `graph.invoke`; an **observation** is a node within it; a **generation** is one LLM call within a node.
- LangChain **tags** are the unit of *per-agent* aggregation. Tag every LLM call with its role and you get cost / latency / token breakdowns for free.
- **Model definitions** in Langfuse map model strings (via regex) to per-token prices. Without a match, costs read `$0`.
- Per-run knobs go in **`PlanContext`** (`improved_main_task`, `available_agents`, `*_append_instructions`, …); the compiled graph never needs to change.

**Code**

- `from cmbagent_lg.tracing import langfuse_handler` — one line of tracing wiring.
- `graph.invoke({}, context=ctx, config={'callbacks': [handler]})` — identical to Lab 3.
- `cmbagent-lg-cost <trace_id | workdir>` — one CLI for cost / latency / per-agent breakdowns.

**Where to go next**

- Port more cmbagent agents (engineer, researcher, plotter, …) as LangGraph nodes that the planner can route to.
- Add **evaluations**: Langfuse supports scoring traces, comparing model versions, and running A/B experiments — see the Evaluations tab in the UI.
- Configure prompt management in Langfuse so prompt edits don't require a code change.